# Продажи УТ: исследуем данные и типовую логику интерактивно

Этот notebook дополняет обзорный сценарий ЗУП и показывает, что подход не привязан к одной конфигурации или предметной области.

На УТ 11.6.1.61 мы найдём последний месяц с данными и предыдущий календарный, сравним выручку в pandas, вызовем типовую функцию расчёта прироста и проверим альтернативную трактовку через hot reload.

Здесь меньше деталей API: цель — компактный независимый end-to-end пример на другой конфигурации.

## Подготовка

Укажите `PLATFORM_BIN`, `CONNECTION_STRING` и `SOURCE_ROOT` для отдельной копии УТ 11.6.1.61. В `SOURCE_ROOT` нужна выгрузка исходников этой же конфигурации.

Как и в ЗУП-сценарии, одна стартовая ячейка открывает сеанс 1С, а дальнейшие BSL- и Python-ячейки работают с общим состоянием.

In [ ]:
from IPython.display import display
from onec_runtime.config import RuntimeConfig
from onec_runtime.session import ExtensionMode, RuntimeSessionConfig
from onec_runtime_jupyter import InteractiveRuntimeSession

PLATFORM_BIN = r'C:\path\to\1cv8\bin'
CONNECTION_STRING = r'File="C:\demo\UT";'
SOURCE_ROOT = r'C:\exports\UT'
EXTENSION_MODE = ExtensionMode.AUTO  # Совместимое расширение проверяется при запуске.

runtime = InteractiveRuntimeSession.start(
    RuntimeSessionConfig(
        runtime=RuntimeConfig(
            platform_bin=PLATFORM_BIN,
            connection_string=CONNECTION_STRING,
        ),
        source_root=SOURCE_ROOT,
        extension_mode=EXTENSION_MODE,
    )
)

## Находим период для опыта

Чтобы notebook не зависел от конкретной даты демобазы, найдём последнее активное движение с ненулевой выручкой. Сравним месяц этой даты с предыдущим календарным месяцем.

In [ ]:
%%bsl
ЗапросПоследнейПродажи = Новый Запрос;
ЗапросПоследнейПродажи.Текст =
    "ВЫБРАТЬ ПЕРВЫЕ 1
    |   Движения.Период КАК Период
    |ИЗ
    |   РегистрНакопления.ВыручкаИСебестоимостьПродаж КАК Движения
    |ГДЕ
    |   Движения.Активность
    |   И Движения.СуммаВыручки <> 0
    |УПОРЯДОЧИТЬ ПО
    |   Движения.Период УБЫВ";
ПоследняяПродажа = ЗапросПоследнейПродажи.Выполнить().Выбрать();
Если Не ПоследняяПродажа.Следующий() Тогда
    ВызватьИсключение "В регистре нет активных движений выручки. Выберите ИБ с продажами.";
КонецЕсли;

НачалоПоследнегоМесяца = НачалоМесяца(ПоследняяПродажа.Период);
НачалоПредыдущегоМесяца = ДобавитьМесяц(НачалоПоследнегоМесяца, -1);
КонецПоследнегоМесяца = КонецМесяца(ПоследняяПродажа.Период);

## Получаем обороты средствами УТ

Используем виртуальную таблицу `Обороты` регистра `ВыручкаИСебестоимостьПродаж`. Она учитывает движения выручки, включая корректировки и возвраты. Группируем данные по месяцу и подразделению.

In [ ]:
%%bsl
ЗапросПродаж = Новый Запрос;
ЗапросПродаж.Текст =
    "ВЫБРАТЬ
    |   Продажи.Период КАК Месяц,
    |   Продажи.Подразделение КАК Подразделение,
    |   СУММА(Продажи.СуммаВыручкиОборот) КАК Выручка
    |ИЗ
    |   РегистрНакопления.ВыручкаИСебестоимостьПродаж.Обороты(
    |       &НачалоПериода, &КонецПериода, Месяц, ) КАК Продажи
    |СГРУППИРОВАТЬ ПО
    |   Продажи.Период,
    |   Продажи.Подразделение
    |УПОРЯДОЧИТЬ ПО
    |   Месяц";
ЗапросПродаж.УстановитьПараметр("НачалоПериода", НачалоПредыдущегоМесяца);
ЗапросПродаж.УстановитьПараметр("КонецПериода", КонецПоследнегоМесяца);
ПродажиПоМесяцам = ЗапросПродаж.Выполнить().Выгрузить();
Если ПродажиПоМесяцам.Количество() = 0 Тогда
    ВызватьИсключение "За выбранные два месяца оборотов выручки нет.";
КонецЕсли;

In [ ]:
ПродажиПоМесяцам[:2].to_df()

## Из 1С в pandas

Переносим результат запроса в pandas и строим несколько простых представлений. Ссылки материализуем как читаемые представления: здесь они нужны для анализа человеком, а не как технические идентификаторы.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sales = ПродажиПоМесяцам.to_df(refs='presentation')
display(sales)
monthly = (sales.assign(Месяц=pd.to_datetime(sales['Месяц']).dt.to_period('M'))
           .groupby('Месяц')['Выручка'].sum())
months = pd.period_range(end=monthly.index.max(), periods=2, freq='M')
monthly = monthly.reindex(months, fill_value=0)
display(monthly.rename_axis('Месяц').to_frame())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(month) for month in monthly.index], monthly.map(float))
ax.set_ylabel('Выручка с НДС')
ax.set_title('Два последних календарных месяца по данным ИБ')
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
latest_month = sales['Месяц'].max()
by_department = (sales.loc[sales['Месяц'] == latest_month]
                 .assign(Подразделение=lambda data:
                         data['Подразделение'].fillna('Без подразделения'))
                 .groupby('Подразделение')['Выручка'].sum()
                 .sort_values(ascending=False).head(10))
display(by_department.to_frame('Выручка'))

fig, ax = plt.subplots(figsize=(9, max(3, len(by_department) * 0.4)))
ax.barh(by_department.index, by_department.map(float))
ax.invert_yaxis()
ax.set_xlabel('Выручка с НДС')
ax.set_title(f'Подразделения за {str(latest_month)[:7]}')
fig.tight_layout()
display(fig)
plt.close(fig)

## Вызываем типовую бизнес-логику

Теперь оставим агрегацию выручки в BSL и вызовем `ПродажиСервер.ПроцентПрироста`.

Интересен не только результат для наших двух месяцев, но и контрольный вызов `(0, 100)`: в УТ 11.6.1.61 функция отдельно обрабатывает нулевую и отрицательную базу сравнения.

In [ ]:
%%bsl
ВыручкаПредыдущегоМесяца = 0;
ВыручкаПоследнегоМесяца = 0;
Для Каждого СтрокаПродаж Из ПродажиПоМесяцам Цикл
    Если СтрокаПродаж.Месяц = НачалоПоследнегоМесяца Тогда
        ВыручкаПоследнегоМесяца = ВыручкаПоследнегоМесяца + СтрокаПродаж.Выручка;
    Иначе
        ВыручкаПредыдущегоМесяца = ВыручкаПредыдущегоМесяца + СтрокаПродаж.Выручка;
    КонецЕсли;
КонецЦикла;

ПриростВыручки = ПродажиСервер.ПроцентПрироста(
    ВыручкаПредыдущегоМесяца, ВыручкаПоследнегоМесяца);
КонтрольныйПриростДо = ПродажиСервер.ПроцентПрироста(0, 100);

In [ ]:
growth_before = ПриростВыручки.materialize()
probe_before = КонтрольныйПриростДо.materialize()
print('Прирост выручки, %:', growth_before)
print('Контрольный вызов (0, 100), %:', probe_before)

## Hot reload: проверяем альтернативную трактовку

Оригинальная функция возвращает 100% для пары `(0, 100)`. Проверим другую бизнес-гипотезу: если предыдущая выручка равна нулю, считать процент прироста неопределённым.

Изменяем только локальную копию `CommonModules/ПродажиСервер/Ext/Module.bsl`, загружаем её в текущий экспериментальный сеанс и повторяем те же вызовы. Конфигурация ИБ при этом не обновляется.

Это лишь короткая демонстрация возможности; устройство hot reload и его ограничения разбираются в отдельном материале.

In [ ]:
runtime.load_worker_module(r'CommonModules\ПродажиСервер\Ext\Module.bsl')

In [ ]:
%%bsl
ПриростПослеПерезагрузки = ПродажиСервер.ПроцентПрироста(
    ВыручкаПредыдущегоМесяца, ВыручкаПоследнегоМесяца);
КонтрольныйПриростПосле = ПродажиСервер.ПроцентПрироста(0, 100);

In [ ]:
growth_before = ПриростВыручки.materialize()
probe_before = КонтрольныйПриростДо.materialize()
print('Прирост выручки, %:', growth_before)
print('Контрольный вызов (0, 100), %:', probe_before)

## Завершение

In [ ]:
runtime.close()
print('Сеанс закрыт')